<a href="https://colab.research.google.com/github/HenryZumaeta/py4cd_EPC2025/blob/main/C23/C23_Script01_Regulaizacion_Lasso_Ridge_ElasticNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión Lineal

In [3]:
# Módulos y datos
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Datos: https://cdn.uci-ics-mlr-prod.aws.uci.edu/183/communities%2Band%2Bcrime.zip
os.system("wget https://cdn.uci-ics-mlr-prod.aws.uci.edu/183/communities%2Band%2Bcrime.zip")

0

In [4]:
# Descomprimimos el archivo zip+
os.system("unzip communities+and+crime.zip")

256

In [5]:
# Mostremos/observemos las primeras líneas de este dataset
!cat communities.data | head -n 5

8,?,?,Lakewoodcity,1,0.19,0.33,0.02,0.9,0.12,0.17,0.34,0.47,0.29,0.32,0.2,1,0.37,0.72,0.34,0.6,0.29,0.15,0.43,0.39,0.4,0.39,0.32,0.27,0.27,0.36,0.41,0.08,0.19,0.1,0.18,0.48,0.27,0.68,0.23,0.41,0.25,0.52,0.68,0.4,0.75,0.75,0.35,0.55,0.59,0.61,0.56,0.74,0.76,0.04,0.14,0.03,0.24,0.27,0.37,0.39,0.07,0.07,0.08,0.08,0.89,0.06,0.14,0.13,0.33,0.39,0.28,0.55,0.09,0.51,0.5,0.21,0.71,0.52,0.05,0.26,0.65,0.14,0.06,0.22,0.19,0.18,0.36,0.35,0.38,0.34,0.38,0.46,0.25,0.04,0,0.12,0.42,0.5,0.51,0.64,0.03,0.13,0.96,0.17,0.06,0.18,0.44,0.13,0.94,0.93,0.03,0.07,0.1,0.07,0.02,0.57,0.29,0.12,0.26,0.2,0.06,0.04,0.9,0.5,0.32,0.14,0.2
53,?,?,Tukwilacity,1,0,0.16,0.12,0.74,0.45,0.07,0.26,0.59,0.35,0.27,0.02,1,0.31,0.72,0.11,0.45,0.25,0.29,0.39,0.29,0.37,0.38,0.33,0.16,0.3,0.22,0.35,0.01,0.24,0.14,0.24,0.3,0.27,0.73,0.57,0.15,0.42,0.36,1,0.63,0.91,1,0.29,0.43,0.47,0.6,0.39,0.46,0.53,0,0.24,0.01,0.52,0.62,0.64,0.63,0.25,0.27,0.25,0.23,0.84,0.1,0.16,0.1,0.17,0.29,0.17,0.26,0.2,0.82,0,0.02,0.79,0.24,0.02,0.25,0.65,0

In [6]:
# Carguemos en memoria el conjunto de datos
df = pd.read_csv("communities.data", header=None, na_values="?")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Columns: 128 entries, 0 to 127
dtypes: float64(125), int64(2), object(1)
memory usage: 1.9+ MB


In [7]:
# Del archivo comunities.names vamos a obtener los nombres de las columnas

columns = []

with open("communities.names", "r") as f:
    for line in f:
        if line.startswith("@attribute"):
            columns.append(line.split()[1])

df.columns = columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Columns: 128 entries, state to ViolentCrimesPerPop
dtypes: float64(125), int64(2), object(1)
memory usage: 1.9+ MB


In [8]:
df.head()

,state,county,community,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,...,LandArea,PopDens,PctUsePubTrans,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,LemasPctOfficDrugUn,PolicBudgPerPop,ViolentCrimesPerPop
0,8,NaN,NaN,Lakewoodcity,1,0.19,0.33,0.02,0.90,0.12,...,0.12,0.26,0.20,0.06,0.04,0.9,0.5,0.32,0.14,0.20
1,53,NaN,NaN,Tukwilacity,1,0.00,0.16,0.12,0.74,0.45,...,0.02,0.12,0.45,NaN,NaN,NaN,NaN,0.00,NaN,0.67
2,24,NaN,NaN,Aberdeentown,1,0.00,0.42,0.49,0.56,0.17,...,0.01,0.21,0.02,NaN,NaN,NaN,NaN,0.00,NaN,0.43
3,34,5.0,81440.0,Willingborotownship,1,0.04,0.77,1.00,0.08,0.12,...,0.02,0.39,0.28,NaN,NaN,NaN,NaN,0.00,NaN,0.12
4,42,95.0,6096.0,Bethlehemtownship,1,0.01,0.55,0.02,0.95,0.09,...,0.04,0.09,0.02,NaN,NaN,NaN,NaN,0.00,NaN,0.03


In [9]:
# Veamos qué columnas tienen valores faltantes
df.isnull().sum().sort_values(ascending=False)

,0
PolicBudgPerPop,1675
LemasGangUnitDeploy,1675
LemasPctPolicOnPatr,1675
PolicCars,1675
PolicOperBudg,1675
...,...
LandArea,0
PopDens,0
PctUsePubTrans,0
LemasPctOfficDrugUn,0


In [10]:
# Primer escenario: Eliminar a las filas con valores faltantes
df.dropna(axis = 0)

,state,county,community,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,...,LandArea,PopDens,PctUsePubTrans,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,LemasPctOfficDrugUn,PolicBudgPerPop,ViolentCrimesPerPop
16,36,1.0,1000.0,Albanycity,1,0.15,0.31,0.40,0.63,0.14,...,0.06,0.39,0.84,0.06,0.06,0.91,0.5,0.88,0.26,0.49
23,19,193.0,93926.0,SiouxCitycity,1,0.11,0.43,0.04,0.89,0.09,...,0.16,0.12,0.07,0.04,0.01,0.81,1.0,0.56,0.09,0.63
33,51,680.0,47672.0,Lynchburgcity,1,0.09,0.43,0.51,0.58,0.04,...,0.14,0.11,0.19,0.05,0.01,0.75,0.0,0.60,0.10,0.31
68,34,23.0,58200.0,PerthAmboycity,1,0.05,0.59,0.23,0.39,0.09,...,0.01,0.73,0.28,0.00,0.02,0.64,0.0,1.00,0.23,0.50
74,9,9.0,46520.0,Meridentown,1,0.08,0.39,0.08,0.85,0.04,...,0.07,0.21,0.04,0.02,0.01,0.70,1.0,0.44,0.11,0.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1880,34,39.0,40350.0,Lindencity,10,0.04,0.39,0.39,0.65,0.09,...,0.03,0.28,0.32,0.02,0.01,0.85,0.0,0.99,0.19,0.22
1963,36,27.0,59641.0,Poughkeepsiecity,10,0.03,0.32,0.61,0.47,0.09,...,0.01,0.47,0.42,0.07,0.08,0.49,0.0,0.37,1.00,0.45
1981,9,9.0,35650.0,Hamdentown,10,0.07,0.38,0.17,0.84,0.11,...,0.09,0.13,0.17,0.02,0.01,0.72,0.0,0.62,0.15,0.07
1991,9,9.0,80070.0,Waterburytown,10,0.16,0.37,0.25,0.69,0.04,...,0.08,0.32,0.18,0.08,0.06,0.78,0.0,0.91,0.28,0.23


In [11]:
# Segundo escenario: Eliminar a las columnas con valores faltantes
df.dropna(axis = 1)

,state,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,...,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans,LemasPctOfficDrugUn,ViolentCrimesPerPop
0,8,Lakewoodcity,1,0.19,0.33,0.02,0.90,0.12,0.17,0.34,...,0.12,0.42,0.50,0.51,0.64,0.12,0.26,0.20,0.32,0.20
1,53,Tukwilacity,1,0.00,0.16,0.12,0.74,0.45,0.07,0.26,...,0.21,0.50,0.34,0.60,0.52,0.02,0.12,0.45,0.00,0.67
2,24,Aberdeentown,1,0.00,0.42,0.49,0.56,0.17,0.04,0.39,...,0.14,0.49,0.54,0.67,0.56,0.01,0.21,0.02,0.00,0.43
3,34,Willingborotownship,1,0.04,0.77,1.00,0.08,0.12,0.10,0.51,...,0.19,0.30,0.73,0.64,0.65,0.02,0.39,0.28,0.00,0.12
4,42,Bethlehemtownship,1,0.01,0.55,0.02,0.95,0.09,0.05,0.38,...,0.11,0.72,0.64,0.61,0.53,0.04,0.09,0.02,0.00,0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1989,12,TempleTerracecity,10,0.01,0.40,0.10,0.87,0.12,0.16,0.43,...,0.22,0.28,0.34,0.48,0.39,0.01,0.28,0.05,0.00,0.09
1990,6,Seasidecity,10,0.05,0.96,0.46,0.28,0.83,0.32,0.69,...,0.53,0.25,0.17,0.10,0.00,0.02,0.37,0.20,0.00,0.45
1991,9,Waterburytown,10,0.16,0.37,0.25,0.69,0.04,0.25,0.35,...,0.25,0.68,0.61,0.79,0.76,0.08,0.32,0.18,0.91,0.23
1992,25,Walthamcity,10,0.08,0.51,0.06,0.87,0.22,0.10,0.58,...,0.45,0.64,0.54,0.59,0.52,0.03,0.38,0.33,0.22,0.19


In [12]:
# Estas son las columnas con valorse faltantes
df.loc[:, df.isnull().sum().sort_values(ascending = False) != 0].columns

Index(['county', 'community', 'OtherPerCap', 'LemasSwornFT', 'LemasSwFTPerPop',
       'LemasSwFTFieldOps', 'LemasSwFTFieldPerPop', 'LemasTotalReq',
       'LemasTotReqPerPop', 'PolicReqPerOffic', 'PolicPerPop',
       'RacialMatchCommPol', 'PctPolicWhite', 'PctPolicBlack', 'PctPolicHisp',
       'PctPolicAsian', 'PctPolicMinor', 'OfficAssgnDrugUnits',
       'NumKindsDrugsSeiz', 'PolicAveOTWorked', 'PolicCars', 'PolicOperBudg',
       'LemasPctPolicOnPatr', 'LemasGangUnitDeploy', 'PolicBudgPerPop'],
      dtype='object')

In [13]:
# Entonces el dataset que usaremos para la regresión es:
df = df.dropna(axis = 1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Columns: 103 entries, state to ViolentCrimesPerPop
dtypes: float64(100), int64(2), object(1)
memory usage: 1.6+ MB


In [14]:
# Separemos las varialbes independientes (predictoras) y la variable dependiente (target)
X = df.drop(columns = ["ViolentCrimesPerPop"])
X = X.select_dtypes(include = ["float64"])
y = df["ViolentCrimesPerPop"]

In [15]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 99 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   population             1994 non-null   float64
 1   householdsize          1994 non-null   float64
 2   racepctblack           1994 non-null   float64
 3   racePctWhite           1994 non-null   float64
 4   racePctAsian           1994 non-null   float64
 5   racePctHisp            1994 non-null   float64
 6   agePct12t21            1994 non-null   float64
 7   agePct12t29            1994 non-null   float64
 8   agePct16t24            1994 non-null   float64
 9   agePct65up             1994 non-null   float64
 10  numbUrban              1994 non-null   float64
 11  pctUrban               1994 non-null   float64
 12  medIncome              1994 non-null   float64
 13  pctWWage               1994 non-null   float64
 14  pctWFarmSelf           1994 non-null   float64
 15  pctW

## Primer Modelo

In [16]:
# Construyamos un modelo de regresión lineal
import statsmodels.api as sm

# Un requerimiento de statsmodels, agregar constante para el intercepto
X = sm.add_constant(X)

# Ajuste de una OLS
model = sm.OLS(y,X)
results = model.fit()
print(results.summary())

                             OLS Regression Results                            
Dep. Variable:     ViolentCrimesPerPop   R-squared:                       0.694
Model:                             OLS   Adj. R-squared:                  0.678
Method:                  Least Squares   F-statistic:                     43.34
Date:                 Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                         23:02:35   Log-Likelihood:                 1255.8
No. Observations:                 1994   AIC:                            -2312.
Df Residuals:                     1894   BIC:                            -1752.
Df Model:                           99                                         
Covariance Type:             nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const               

In [17]:
# Obtengamos los p-values de cada variable en model
results.pvalues

,0
const,0.006028
population,0.621230
householdsize,0.783542
racepctblack,0.000061
racePctWhite,0.450534
...,...
PctSameState85,0.709235
LandArea,0.604185
PopDens,0.727348
PctUsePubTrans,0.082692


In [18]:
# Obtener los nombres de las variables cuyo p-values sea menor a 5%
#significant_vars = results.pvalues[results.pvalues < 0.05].index
summary_table = pd.DataFrame({
    "coef": results.params,
    "pvalue": results.pvalues
})

significant = summary_table[summary_table["pvalue"] < 0.05]
significant

,coef,pvalue
const,0.559274,0.006028
racepctblack,0.206004,0.000061
pctUrban,0.047656,0.002377
pctWWage,-0.199779,0.025682
pctWFarmSelf,0.046859,0.020308
pctWInvInc,-0.172693,0.010807
pctWRetire,-0.090352,0.014422
whitePerCap,-0.340850,0.025729
HispPerCap,0.050158,0.035201
PctPopUnderPov,-0.180034,0.004204


In [19]:
# Variables significativas
significant_vars = list(significant.index)
len(significant_vars)

28

## Segundo Modelo

Usando las variables significativas del primer modelo

In [20]:
# Seleccionemos solo las variables significativas
X_reduced = X[significant_vars]

# Ajustemos un segundo modelo de regresión lineal (reducido)
model_reduced = sm.OLS(y, X_reduced)
results_reduced = model_reduced.fit()
print(results_reduced.summary())

                             OLS Regression Results                            
Dep. Variable:     ViolentCrimesPerPop   R-squared:                       0.673
Model:                             OLS   Adj. R-squared:                  0.668
Method:                  Least Squares   F-statistic:                     149.7
Date:                 Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                         23:02:49   Log-Likelihood:                 1189.6
No. Observations:                 1994   AIC:                            -2323.
Df Residuals:                     1966   BIC:                            -2167.
Df Model:                           27                                         
Covariance Type:             nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const               

In [21]:
# Comparemos los resultados de estos modelos
print("R² Original: ", results.rsquared)
print("R² reducido: ", results_reduced.rsquared)
print()
print("Adj R² Original: ", results.rsquared_adj)
print("Adj R² reducido: ", results_reduced.rsquared_adj)
print()
print("AIC Original: ", results.aic)
print("AIC reducido: ", results_reduced.aic)
print()
print("BIC Original: ", results.bic)
print("BIC reducido: ", results_reduced.bic)

R² Original:  0.6937465628945809
R² reducido:  0.6727506862582984

Adj R² Original:  0.6777385954851636
Adj R² reducido:  0.6682564179617441

AIC Original:  -2311.503042965238
AIC reducido:  -2323.282253412701

BIC Original:  -1751.7132479130596
BIC reducido:  -2166.541110798091


In [ ]:
# R² Original:  0.6937465628945809
# R² reducido:  0.6727506862582984

# La pérdida de poder explicativo (69.4 - 67.3) aprox. 2%, es muy pequeña a pesar
# de que se ha eliminado una gran cantidad de variables (99-27 = 72).

# Esto nos dice que muchas variables del modelo orifinal, aportaban muy poca información

In [ ]:
# AIC Original:  -2311.503042965238
# AIC reducido:  -2323.282253412701

# REGLA: Menor AIC ==> "Mejor modelo"

# Conclusión: El modelo reducido es preferible según AIC.
# Esto es, que el equilibrio entre el ajuste del modelo y la complejidad es mejor
# en el modelo reducido.

In [ ]:
# BIC Original:  -1751.7132479130596
# BIC reducido:  -2166.541110798091

# REGLA: Menor BIC ==> "Mejor modelo"

# El modelo reducido tiene un BIC mucho menor, lo cual es indicativo que el modelo
# reducido tiene una mejora muy fuerte en parsimonía ==> Lo cual implica que hay
# un menor riesgo de sobreajuste.

In [ ]:
# EN CONCLUSIÓN:
# --------------
    # Usa 27 variables en lugar de 99
    # Mantiene casi el mismo poder explicativo (R²)
    # Mejora AIC
    # Mejora BIC
# Esto nos dice que el modelo reducido es un modelo estadísticamente más
# eficiente.

# Se da el principio de OCCAM: Dos modelos con la capacidad predictiva similar,
# preferimos el modelo más simple.

In [22]:
# Coeficientes y p-values del modelo original
significant

,coef,pvalue
const,0.559274,0.006028
racepctblack,0.206004,0.000061
pctUrban,0.047656,0.002377
pctWWage,-0.199779,0.025682
pctWFarmSelf,0.046859,0.020308
pctWInvInc,-0.172693,0.010807
pctWRetire,-0.090352,0.014422
whitePerCap,-0.340850,0.025729
HispPerCap,0.050158,0.035201
PctPopUnderPov,-0.180034,0.004204


In [23]:
# Obtener los nombres de las variables cuyo p-values
summary_table_reduced = pd.DataFrame({
    "coef": results_reduced.params,
    "pvalue": results_reduced.pvalues
})

significant_reduced = summary_table_reduced.copy()
significant_reduced

,coef,pvalue
const,0.723168,4.367029e-31
racepctblack,0.183841,9.832202e-11
pctUrban,0.033576,1.519178e-04
pctWWage,-0.235063,5.173692e-06
pctWFarmSelf,0.028384,1.357603e-01
pctWInvInc,-0.108347,3.037171e-02
pctWRetire,-0.114891,2.431367e-04
whitePerCap,-0.099404,2.135645e-02
HispPerCap,0.042511,6.379943e-02
PctPopUnderPov,-0.179640,1.658285e-05


# Regularización Lasso (L1)
 usando sklearn

In [24]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv(
    "communities.data",
    header = None,
    na_values = "?"
)

columns = []
with open("communities.names") as f:
  for line in f:
    if line.startswith("@attribute"):
      columns.append(line.split()[1])
df.columns = columns
df = df.dropna(axis = 1)
X = df.drop(columns = ["ViolentCrimesPerPop"])
X = X.select_dtypes(include = ["float64"])
y = df["ViolentCrimesPerPop"]

In [25]:
# Estandarización (Obligatorio para Lasso)
scale = StandardScaler()
X_scaled_data = scale.fit_transform(X)

# Convertir X_scaled_data a un dataframe para mantener los nombres de las columnas
X_scaled = pd.DataFrame(X_scaled_data, columns = X.columns, index = X.index)

# Lista de valores alpha a probar
mis_alphas = np.logspace(-4, 1, 20)

# Consideremos 10 validaciones cruzadas
lasso_cv = LassoCV(alphas = mis_alphas,
                   cv = 10,
                   max_iter =10000,
                   tol = 1e-5,
                   random_state = 666,
                   fit_intercept=True)

# Entrenamos el modelo
lasso_cv.fit(X_scaled, y)

LassoCV(alphas=array([1.00000000e-04, 1.83298071e-04, 3.35981829e-04, 6.15848211e-04,
       1.12883789e-03, 2.06913808e-03, 3.79269019e-03, 6.95192796e-03,
       1.27427499e-02, 2.33572147e-02, 4.28133240e-02, 7.84759970e-02,
       1.43844989e-01, 2.63665090e-01, 4.83293024e-01, 8.85866790e-01,
       1.62377674e+00, 2.97635144e+00, 5.45559478e+00, 1.00000000e+01]),
        cv=10, max_iter=10000, random_state=666, tol=1e-05)

In [26]:
# dir(lasso_cv)
# El mejor alpha
best_alpha = lasso_cv.alpha_
best_alpha

np.float64(0.0003359818286283781)

In [27]:
# Coeficientes del modelo
coeficients = lasso_cv.coef_
intercept = lasso_cv.intercept_

# Crearemos un dataframe para visualizar los coeficientes
results_df = pd.DataFrame({
    "Variable": X.columns.tolist(),
    "Coeficiente": coeficients

})

results_df.sort_values(by = "Coeficiente", ascending = False)


,Variable,Coeficiente
63,PersPerOccupHous,0.051979
2,racepctblack,0.049238
84,MedRent,0.048188
67,PctPersDenseHous,0.033957
49,PctIlleg,0.033152
...,...,...
13,pctWWage,-0.028205
27,PctPopUnderPov,-0.031628
7,agePct12t29,-0.035632
81,RentLowQ,-0.042762


In [28]:
# Veamos aquellas variables cuyo coeficiente es cero (Lasso)
results_df[results_df["Coeficiente"] == 0].shape

# La técnica de regularización Lasso a vuelto 29 coeficientes

(29, 2)

In [29]:
# Filtrar las variables seleccionadas (Coeficiente != 0)
selected_var = results_df[results_df["Coeficiente"] != 0].sort_values(by = "Coeficiente", ascending = False, key = abs)
selected_var

,Variable,Coeficiente
43,PctKids2Par,-0.055850
63,PersPerOccupHous,0.051979
2,racepctblack,0.049238
84,MedRent,0.048188
81,RentLowQ,-0.042762
...,...,...
77,PctWOFullPlumb,-0.001975
9,agePct65up,0.001734
75,MedYrHousBuilt,-0.001654
51,PctImmigRecent,0.000929


In [30]:
# Evaluación del rendimiento
y_pred = lasso_cv.predict(X_scaled)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("Métricas de Ajuste - Lasso:")
print(f"R2: {r2:.6f}")
print(f"MSE: {mse*100:.6f}%")
#

Métricas de Ajuste - Lasso:
R2: 0.688740
MSE: 1.688734%


# Regularización Ridge (L2)

In [31]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv(
    "communities.data",
    header = None,
    na_values = "?"
)

columns = []
with open("communities.names") as f:
  for line in f:
    if line.startswith("@attribute"):
      columns.append(line.split()[1])
df.columns = columns
df = df.dropna(axis = 1)
X = df.drop(columns = ["ViolentCrimesPerPop"])
X = X.select_dtypes(include = ["float64"])
y = df["ViolentCrimesPerPop"]

In [32]:
# Estandarización (Obligatorio para Ridge)
scale = StandardScaler()
X_scaled_data = scale.fit_transform(X)

# Convertir X_scaled_data a un dataframe para mantener los nombres de las columnas
X_scaled = pd.DataFrame(X_scaled_data, columns = X.columns, index = X.index)

# Lista de valores alpha a probar
mis_alphas = np.logspace(-4, 1, 20)

# Consideremos 10 validaciones cruzadas
ridge_cv = RidgeCV(alphas = mis_alphas,
                   cv = 10)

# Entrenamos el modelo
ridge_cv.fit(X_scaled, y)

RidgeCV(alphas=array([1.00000000e-04, 1.83298071e-04, 3.35981829e-04, 6.15848211e-04,
       1.12883789e-03, 2.06913808e-03, 3.79269019e-03, 6.95192796e-03,
       1.27427499e-02, 2.33572147e-02, 4.28133240e-02, 7.84759970e-02,
       1.43844989e-01, 2.63665090e-01, 4.83293024e-01, 8.85866790e-01,
       1.62377674e+00, 2.97635144e+00, 5.45559478e+00, 1.00000000e+01]),
        cv=10)

In [33]:
# El mejor alpha
best_alpha = ridge_cv.alpha_
best_alpha

np.float64(10.0)

In [34]:
# Coeficientes del modelo
coeficients = ridge_cv.coef_
intercept = ridge_cv.intercept_

# Crearemos un dataframe para visualizar los coeficientes
results_df = pd.DataFrame({
    "Variable": X.columns.tolist(),
    "Coeficiente": coeficients

})

results_df.sort_values(by = "Coeficiente", ascending = False)

,Variable,Coeficiente
63,PersPerOccupHous,0.065419
84,MedRent,0.056120
2,racepctblack,0.049428
67,PctPersDenseHous,0.040061
37,MalePctDivorce,0.037015
...,...,...
7,agePct12t29,-0.036558
66,PctPersOwnOccup,-0.038037
21,whitePerCap,-0.044376
81,RentLowQ,-0.045685


In [38]:
# Veamos los coeficientes más pequeños en valor absoluto
results_df.reindex(results_df["Coeficiente"].abs().sort_values().index).head(10)

,Variable,Coeficiente
0,population,0.000161
96,PopDens,-0.001066
52,PctImmigRec5,-0.001204
45,PctTeen2Par,-0.001227
91,PctBornSameState,-0.001266
12,medIncome,-0.001398
1,householdsize,0.001402
58,PctRecImmig10,-0.001445
92,PctSameHouse85,-0.001680
17,pctWPubAsst,0.001680


In [36]:
# Filtrar las variables seleccionadas (Coeficiente != 0)
selected_var = results_df[results_df["Coeficiente"] != 0].sort_values(by = "Coeficiente", ascending = False, key = abs)
selected_var

,Variable,Coeficiente
63,PersPerOccupHous,0.065419
84,MedRent,0.056120
2,racepctblack,0.049428
43,PctKids2Par,-0.047380
81,RentLowQ,-0.045685
...,...,...
91,PctBornSameState,-0.001266
45,PctTeen2Par,-0.001227
52,PctImmigRec5,-0.001204
96,PopDens,-0.001066


In [39]:
# Evaluación del rendimiento
y_pred = ridge_cv.predict(X_scaled)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("Métricas de Ajuste - Ridge:")
print(f"R2: {r2:.6f}")
print(f"MSE: {mse*100:.6f}%")

Métricas de Ajuste - Ridge:
R2: 0.691819
MSE: 1.672026%


# Regularización ElasticNet

https://hastie.su.domains/Papers/elasticnet.pdf

In [43]:
# Para las Validaciones Cruzadas (CV)
from sklearn.model_selection import KFold

df = pd.read_csv(
    "communities.data",
    header = None,
    na_values = "?"
)

columns = []

with open("communities.names") as f:
  for line in f:
    if line.startswith("@attribute"):
      columns.append(line.split()[1])

df.columns = columns

df = df.dropna(axis = 1)

X = df.drop(columns = ["ViolentCrimesPerPop"])
X = X.select_dtypes(include = ["float64"])
y = df["ViolentCrimesPerPop"]

In [42]:
# Estandarización e Intercepto
scaler = StandardScaler()
X_scaled_data = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_data, columns = X.columns, index = X.index)

# Añadimos la constante para el intercepto: Requerimiento para statsmodels
X_scaled_const = sm.add_constant(X_scaled)

# Búsqueda de los mejores hiperparámetros
def find_best_elastic_net_params(X, y, alphas, l1_weights, n_splits = 5):
    kf = KFold(n_splits = n_splits, shuffle = True, random_state = 666)

    best_alpha = None
    best_l1_wt = None
    min_mse = float("inf")

    print("Buscando mejores parámetros (alpha y l1_ratio) mediante CV")

    # Estrategia de Grid Search
    for alpha in alphas:
        for l1_wt in l1_weights:
            mse_scores = []

            for train_idx, test_idx in kf.split(X):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                # Escalado dentro de cada fold
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                X_train_scaled = pd.DataFrame(X_train_scaled, columns = X.columns, index = X_train.index)
                X_test_scaled = pd.DataFrame(X_test_scaled, columns = X.columns, index = X_test.index)

                # Añadimos constante
                X_train_const = sm.add_constant(X_train_scaled, has_constant = "add")
                X_test_const = sm.add_constant(X_test_scaled, has_constant = "add")

                try:
                    # Definir modelo sobre el fold de entrenamiento
                    modelo = sm.OLS(y_train, X_train_const)

                    # Ajustar elastic net
                    results = modelo.fit_regularized(
                        method = "elastic_net",
                        alpha = alpha,
                        L1_wt = l1_wt,
                        maxiter = 10000
                    )

                    y_pred = results.predict(X_test_const)
                    mse = mean_squared_error(y_test, y_pred)
                    mse_scores.append(mse)

                except Exception:
                    mse_scores.append(float("inf"))

            avg_mse = np.mean(mse_scores)

            if avg_mse < min_mse:
                min_mse = avg_mse
                best_alpha = alpha
                best_l1_wt = l1_wt

    return best_alpha, best_l1_wt, min_mse

In [ ]:
# Definamos rangos de búsqueda
alphas_grid = np.logspace(-4, 1, 20)
l1_weights_grid = np.linspace(0, 1, num = 10)

best_alpha, best_l1_wt, cv_mse = find_best_elastic_net_params(X, y, alphas_grid, l1_weights_grid)

print("Mejor alpha:", best_alpha)
print("Mejor L1_wt:", best_l1_wt)
print("CV MSE:", cv_mse)

Buscando mejores parámetros (alpha y l1_ratio) mediante CV


In [ ]:
# Estandarización final
scaler = StandardScaler()
X_scaled_data = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_data, columns = X.columns, index = X.index)

# Añadimos la constante para el intercepto
X_scaled_const = sm.add_constant(X_scaled, has_constant = "add")

# Entrenamiento final de ElasticNet
model_en = sm.OLS(y, X_scaled_const)
result_en = model_en.fit_regularized(
    method = "elastic_net",
    alpha = best_alpha,
    L1_wt = best_l1_wt,
    maxiter = 10000
)

result_en.params

In [ ]:
# Entrenamiento final de ElasticNet
model_en = sm.OLS(y, X_scaled_const)
result_en = model_en.fit_regularized(
    method = "elastic_net",
    alpha = best_alpha,
    L1_wt = best_l1_wt,
    maxiter = 10000
)

result_en.summary()

In [ ]:
coef_df = pd.DataFrame({
    "Variable": X_scaled_const.columns,
    "Coeficiente": result_en.params
})

coef_df.sort_values(by = "Coeficiente", key = abs, ascending = False)

In [ ]:
y_pred = result_en.predict(X_scaled_const)

mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("Métricas de Ajuste - Elastic Net:")
print(f"R2: {r2:.6f}")
print(f"MSE: {mse*100:.6f}%")